# Core Quickstart

This notebook shows the smallest useful EpiScope loop: create structured paper sections, index them locally, retrieve relevant evidence, and return an answer with provenance.


## Setup

Every notebook starts by importing `episcope_nb`, a small helper module that
lives next to the notebooks. It adds `src/` to `sys.path` when it detects a
checkout (installed-package users do not need that), and provides the
deterministic offline stand-ins these notebooks use in place of a real
embedding model and a real LLM.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Locate `notebooks/episcope_nb.py`, the shared helper module. This works whether
# the kernel starts in `notebooks/` or at the repository root.
_cwd = Path.cwd()
_nb_dir = next(
    (
        directory
        for candidate in [_cwd, *_cwd.parents]
        for directory in (candidate, candidate / "notebooks")
        if (directory / "episcope_nb.py").is_file()
    ),
    None,
)
if _nb_dir is None:
    raise FileNotFoundError("Could not find notebooks/episcope_nb.py")
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

# Importing the helper also puts `src/` on sys.path when this is a checkout.
import episcope_nb as nb

WORK_DIR = nb.bootstrap()
WORK_DIR

## Create Example Papers

EpiScope works with `PaperMetadata` plus a list of `StructuredSection` objects. Real ingestion notebooks build these objects from PDFs; here they are written directly so the core API is easy to see.


In [ ]:
from episcope.schemas import PaperMetadata, StructuredSection

sample_papers = {
    "paper_open_data": {
        "metadata": PaperMetadata(
            title="Trial data sharing and reuse",
            abstract="A randomized trial reused a public patient-level dataset from a hospital registry.",
            keywords=["trial", "registry", "data sharing"],
        ),
        "sections": [
            StructuredSection(
                title="Methods",
                section_type="Methods",
                content=(
                    "The analysis used patient records from the National Hospital Registry. "
                    "The registry stores admission dates, treatment groups, and mortality outcomes."
                ),
            ),
            StructuredSection(
                title="Data availability",
                section_type="Data availability",
                content=(
                    "De-identified trial data and the analysis code are available from the public repository. "
                    "The dataset can be reused for non-commercial research after registration."
                ),
            ),
        ],
    },
    "paper_closed_data": {
        "metadata": PaperMetadata(
            title="Hospital cohort study",
            abstract="A cohort study collected clinical data directly from participating hospitals.",
            keywords=["cohort", "hospital", "mortality"],
        ),
        "sections": [
            StructuredSection(
                title="Participants",
                section_type="Methods",
                content=(
                    "The cohort included adult patients admitted to three hospitals. "
                    "Data were collected by the study team from electronic health records."
                ),
            ),
            StructuredSection(
                title="Data sharing",
                section_type="Data availability",
                content=(
                    "The patient dataset cannot be shared publicly because the consent agreement "
                    "does not permit redistribution of individual-level hospital records."
                ),
            ),
        ],
    },
}

list(sample_papers)


## Index And Retrieve

`FileDB` is a local vector store. The tiny embedder keeps the example offline;
replace it with a production embedder for real retrieval quality.

In [ ]:
# A bag-of-keywords embedder: one dimension per vocabulary term. It is
# deterministic and needs no download, so retrieval below is reproducible.
# Real code calls `EmbedderFactory.get_embedder(model_name)` instead.
embedder = nb.TinyKeywordEmbedder()
embedder.model_name, embedder.dim

In [ ]:
from episcope.rag.indexing.chunking import FixedSizeChunker
from episcope.rag.indexing.indexer import Indexer
from episcope.rag.retrieval.candidates import SemanticCandidateRetriever
from episcope.rag.retrieval.retriever import Retriever
from episcope.vectordb.file import FileDB

vdb = FileDB(str(WORK_DIR / "index"))
indexer = Indexer(
    vdb,
    embedder=embedder,
    chunker=FixedSizeChunker(chunk_size=500, chunk_overlap=50),
)

for paper_id, paper in sample_papers.items():
    indexer.index_paper(paper["sections"], paper["metadata"], paper_id=paper_id)

vdb.save()
semantic_candidates = SemanticCandidateRetriever(vdb, dense_embedder=embedder)
retriever = Retriever(vdb, candidate_retrievers=[semantic_candidates], use_rerank=False)

len(vdb.get_points()), vdb.get_embedding_model()


In [ ]:
query = "Which papers use a registry or dataset as a data source?"
results = retriever.retrieve(query, top_k=3)

for rank, result in enumerate(results, start=1):
    print(f"{rank}. {result.paper_id} | {result.section_title} | score={result.similarity_score:.3f}")
    print(result.text[:220], "\n")


## Generate A Traceable Answer

`NoLLMGenerator` simply concatenates retrieved text. It is useful for checking that retrieval and provenance wiring work before adding an LLM client.


In [ ]:
from episcope.rag.generation.nollm_generator import NoLLMGenerator

provenance = NoLLMGenerator().generate(results, question=query)
print(provenance.answer[:700])
print("\nEvidence:")
for evidence in provenance.evidences:
    print(f"- {evidence.paper_id}: {evidence.section}")


You now have the core pattern used throughout the package: structured paper data, a vector store, a retriever, and a generator that returns provenance.
